In [1]:
import os
import warnings
import zipfile

import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [3]:
sensor_loc = pd.read_csv(path + r"5. Source & Refrence Files\sensor_loc.csv",
                         dtype={"station_id": 'str'})
state_map = pd.read_csv(path + r"5. Source & Refrence Files\State_mapping.csv")


In [11]:
folder = path + r"5. Source & Refrence Files\2024_traffic_data"
out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
os.makedirs(out_dir, exist_ok=True)

# build mapping dict ONCE from your state_map df
state_dict = state_map.set_index("state_code")["State"].to_dict()

id_var_col = [
    'record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
    'travel_lane', 'year_record', 'month_record', 'day_record',
    'day_of_week', 'restrictions'
]

part = 0

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)

    if not filename.lower().endswith(".zip"):
        continue

    print(f"Opening ZIP: {filename}")

    with zipfile.ZipFile(file_path, 'r') as z:
        for inner_name in z.namelist():
            if inner_name.endswith("/"):
                continue

            print(f"  Processing inside ZIP: {inner_name}")

            with z.open(inner_name) as f:
                # Read CSV from inside the ZIP
                df = pd.read_csv(
                    f,
                    delimiter="|",
                    low_memory=False  # avoids dtype warning at cost of some RAM, OK for chunk
                    # You can also pass dtype={} here if you know them
                )

                # Melt wide hours columns into long format
                df = pd.melt(
                    df,
                    id_vars=id_var_col,
                    var_name="hours",
                    value_name="traffic"
                )

                # Add state_name via map instead of big merge later
                df["State"] = df["state_code"].map(state_dict)

                # Save this chunk to Parquet and drop from memory
                out_path = os.path.join(out_dir, f"traffic_part_{part}.parquet")
                df.to_parquet(out_path, index=False)
                print(f"    → wrote {out_path}")

                del df
                part += 1

print(f"Done. Wrote {part} parquet files to {out_dir}")


Opening ZIP: apr_2024_ccs_data.zip
  Processing inside ZIP: AK_APR_2024 (TMAS).VOL
    → wrote C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\5. Source & Refrence Files\2024_traffic_parquet\traffic_part_0.parquet
  Processing inside ZIP: AL_APR_2024 (TMAS).VOL
    → wrote C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\5. Source & Refrence Files\2024_traffic_parquet\traffic_part_1.parquet
  Processing inside ZIP: AR_APR_2024 (TMAS).VOL
    → wrote C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\5. Source & Refrence Files\2024_traffic_parquet\traffic_part_2.parquet
  Processing inside ZIP: AZ_APR_2024 (TMAS).VOL
    → wrote C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\5. Source & Refrence Files\2024_traffic_parquet\traffic_part_3.parquet
  Process

In [4]:
# folder = path + r"5. Source & Refrence Files\2024_traffic_data"
#
# dfs = []
#
# for filename in os.listdir(folder):
#     file_path = os.path.join(folder, filename)
#
#     # --- Case 1: ZIP file ---
#     if filename.lower().endswith(".zip"):
#         print(f"Opening ZIP: {filename}")
#
#         with zipfile.ZipFile(file_path, 'r') as z:
#             for inner_name in z.namelist():
#                 # skip folders inside zip
#                 if inner_name.endswith("/"):
#                     continue
#
#                 # print("  Reading inside ZIP:", inner_name)
#
#                 with z.open(inner_name) as f:
#                     df = pd.read_csv(f, delimiter="|")
#                     # print(df.columns)
#                     id_var_col = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
#                    'travel_lane', 'year_record', 'month_record', 'day_record',
#                    'day_of_week', 'restrictions']
#                     df = pd.melt(df, id_vars=id_var_col, var_name='hours', value_name='traffic')
#                     # break
#                     dfs.append(df)
#
# # Combine all data
# combined_traffic_df = pd.concat(dfs, ignore_index=True)

Opening ZIP: apr_2024_ccs_data.zip
Opening ZIP: aug_2024_ccs_data.zip
Opening ZIP: dec_2024_ccs_data.zip
Opening ZIP: feb_2024_ccs_data.zip
Opening ZIP: jan_2024_ccs_data.zip
Opening ZIP: jul_2024_ccs_data.zip
Opening ZIP: jun_2024_ccs_data.zip
Opening ZIP: mar_2024_ccs_data.zip
Opening ZIP: may_2024_ccs_data.zip
Opening ZIP: nov_2024_ccs_data.zip
Opening ZIP: oct_2024_ccs_data.zip
Opening ZIP: sep_2024_ccs_data.zip


In [5]:
columns = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
           'travel_lane', 'year_record', 'month_record', 'day_record',
           'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
           'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
           'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
           'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
           'hour_23', 'restrictions', 'State']

In [6]:
state_dict = state_map.set_index("state_code")["State"].to_dict()
# state_dict

In [7]:
# print(combined_traffic_df.shape)
# combined_traffic_df = pd.merge(combined_traffic_df, state_map, on="state_code", how="left")
# print(combined_traffic_df.shape)
combined_traffic_df["State"] = combined_traffic_df["state_code"].map(state_dict)
combined_traffic_df["station_id"] = combined_traffic_df["station_id"].astype(str)

MemoryError: Unable to allocate 1.40 GiB for an array with shape (187546176,) and data type uint64

In [7]:
traffic_df = pd.DataFrame()

In [8]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 809019


In [9]:
combined_traffic_df["station_id"] = combined_traffic_df["station_id"].str.lstrip("0")

In [10]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 668


In [11]:
combined_traffic_df["station_id"].unique()

array(['E131'], dtype=object)

In [12]:
traffic_df.columns

Index(['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
       'travel_lane', 'year_record', 'month_record', 'day_record',
       'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
       'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
       'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
       'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
       'hour_23', 'restrictions', 'State', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Latitude', 'Longitude', 'Functional Class',
       'Station Id'],
      dtype='object')

In [18]:
clean_col = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
             'travel_lane', 'year_record', 'month_record', 'day_record',
             'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
             'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
             'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
             'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
             'hour_23', 'restrictions', 'State', 'Latitude', 'Longitude', 'Functional Class',
             'Station Id']

In [19]:
traffic_df = traffic_df[clean_col].copy()

In [20]:
id_var_col = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
              'travel_lane', 'year_record', 'month_record', 'day_record',
              'day_of_week', 'restrictions', 'State', 'Latitude', 'Longitude', 'Functional Class',
              'Station Id']

In [21]:
traffic_melt_df = pd.melt(traffic_df, id_vars=id_var_col, var_name='hours', value_name='traffic')

MemoryError: Unable to allocate 16.8 GiB for an array with shape (12, 187530144) and data type object